# RAG Evaluation

This notebook evaluates the complaint chatbot's Retrieval-Augmented Generation (RAG)
pipeline against a set of representative questions and produces a **report-ready**
summary of its strengths and weaknesses.

**Steps**
1. Load the persisted vector store and build the RAG pipeline.
2. Run the representative evaluation questions.
3. Inspect example retrieved chunks and generated answers.
4. Render a markdown evaluation table.
5. Summarize system strengths and weaknesses.

> Prerequisite: run `notebooks/chunking_embedding.ipynb` first so the vector store exists
> in `vector_store/`.

In [ ]:
import sys
from pathlib import Path

from IPython.display import Markdown, display

# Make the project's src package importable from the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.rag import build_rag_pipeline
from src.evaluation import (
    EVALUATION_QUESTIONS,
    run_evaluation,
    to_markdown_table,
    save_markdown_report,
)

PERSIST_DIR = str(PROJECT_ROOT / "vector_store")
BACKEND = "chroma"   # or "faiss"
TOP_K = 5

print(f"Vector store: {PERSIST_DIR}")
print(f"{len(EVALUATION_QUESTIONS)} evaluation questions loaded.")

## 1. Load the vector store and build the RAG pipeline

`build_rag_pipeline` opens the persisted store, wires up the retriever (same embedding
model used at indexing time) and the generator (LLM), and returns a ready-to-use pipeline.

In [ ]:
# Build the pipeline. First run will download the embedding + LLM models.
pipeline = build_rag_pipeline(
    backend=BACKEND,
    persist_dir=PERSIST_DIR,
    top_k=TOP_K,
)
print("RAG pipeline ready.")

## 2. Inspect example retrieved chunks and answers

Before the full table, we look closely at a couple of questions to sanity-check **what the
retriever surfaces** and **how the generator uses it** — including the metadata attached to
each retrieved chunk.

In [ ]:
def show_example(question: str, n_preview: int = 3, chars: int = 300):
    """Display the answer and the top retrieved chunks for one question."""
    result = pipeline.answer(question)
    print("=" * 100)
    print(f"QUESTION: {question}\n")
    print(f"ANSWER:\n{result['answer']}\n")
    print(f"TOP {min(n_preview, len(result['sources']))} RETRIEVED CHUNKS:")
    for i, chunk in enumerate(result["sources"][:n_preview], start=1):
        meta = chunk.metadata
        tag = f"{meta.get('product', '')} | {meta.get('company', '')}".strip(" |")
        print(f"\n  [{i}] score={chunk.score:.3f}  ({tag})")
        print(f"      {chunk.text[:chars]}{'...' if len(chunk.text) > chars else ''}")
    return result

_ = show_example(EVALUATION_QUESTIONS[0])
_ = show_example(EVALUATION_QUESTIONS[-1])  # the out-of-scope control question

## 3. Run all evaluation questions

We run every representative question through the pipeline and collect the answers and
retrieved sources for the evaluation table.

In [ ]:
results = run_evaluation(pipeline, EVALUATION_QUESTIONS)
print(f"Evaluated {len(results)} questions.\n")
for r in results:
    print(f"- Q: {r.question}\n  A: {r.answer[:160]}{'...' if len(r.answer) > 160 else ''}\n")

## 4. Evaluation table

The table below is rendered as markdown so it can be **pasted directly into the final
report**. Fill in the **Quality Score** (1–5) and **Comments** columns after reviewing each
answer against its retrieved sources.

In [ ]:
# Optional: assign quality scores (1-5) and comments after reviewing the answers.
# Edit this dict by question index (0-based); leave blank to fill in manually later.
manual_review = {
    # 0: (4, "Accurate, well grounded in retrieved complaints"),
    # 7: (5, "Correctly refused the out-of-scope question"),
}
for idx, (score, comment) in manual_review.items():
    results[idx].quality_score = score
    results[idx].comments = comment

display(Markdown(to_markdown_table(results)))

In [ ]:
# Save the full report (table + analysis) for pasting into the final deliverable.
report_path = PROJECT_ROOT / "notebooks" / "evaluation_report.md"
report = save_markdown_report(results, path=str(report_path))
print(f"Saved evaluation report -> {report_path}")

## 5. System strengths and weaknesses

> _Refine these bullets after reviewing the table above on your data version._

### Strengths
- **Grounded answers:** responses are constrained to retrieved complaint excerpts, reducing hallucination.
- **Relevant retrieval:** semantic search with `all-MiniLM-L6-v2` surfaces on-topic complaints for in-scope product questions.
- **Safe refusal:** the pipeline returns an explicit "not enough information" message for out-of-scope questions (e.g., the weather control question).
- **Traceability:** every retrieved chunk carries metadata (product, company, issue, date), so answers are auditable.

### Weaknesses
- **Generic phrasing:** answers can be high-level when retrieved chunks are short or only loosely related.
- **Retrieval precision:** some top-k chunks may be tangential; a re-ranking step or smaller chunks could sharpen relevance.
- **Single-product bias:** for broad questions, retrieval may over-represent the most common product in the index.
- **No aggregation:** the model summarizes a few excerpts rather than reasoning over the full volume of complaints.

### Recommended improvements
- Add a cross-encoder **re-ranker** over the top-k results.
- Tune **`top_k`** and the **chunk size/overlap** based on the scores in the table.
- Experiment with a stronger **LLM** for synthesis and a stricter prompt for concision.